In [ ]:
!sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
!jupyter labextension enable

# Imports

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# %%
import datetime
import logging
import os

import pandas as pd
# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option('future.no_silent_downcasting', True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hpandas as hpandas
import helpers.hprint as hprint
import helpers.hcache as hcache

#hcache.get_global_cache_info()
#hcache.clear_global_cache("all")

import config_root.config as cconfig

# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-1a645f79-e931-4409-80c3-244b3777e22a.json'
INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='50f413699'
  # Last commits:
    *   50f413699 GP Saggese Merge branch 'master' into CmampTask11020_Compute_yamm_stats      (60 minutes ago) Wed Jan 8 23:11:29 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    |\  
    * | 91b7ff63d GP Saggese Update                                                            (60 minutes ago) Wed Jan 8 23:11:23 2025           
    | * b682dff49 Dan      Cm task11040 remove get universe from im client 8 (#11164)        (   4 hours ago) Wed Jan 8 20:27:04 2025  (origin/master, origin/HEAD, master)
# Machine info
  system=Linux
  node name=a835e9cd7a0d
  release=6.6.22-linuxkit
  version=#1 SMP Fri Mar 29 12:21:27 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=No

In [9]:
import gspread
print(gspread.__version__)

import gspread_pandas
print(gspread_pandas.__version__)

#gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

#!sudo /bin/f bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

import importlib
import ck_marketing.process_automation.hyamm as hyamm
importlib.reload(hyamm)

import ck_marketing.hunterio.hunter_api as cmhuhuap
importlib.reload(cmhuhuap)

#import helpers.hopenai as hopenai

5.12.4
3.3.0
gspread-gp


<module 'ck_marketing.hunterio.hunter_api' from '/app/ck_marketing/hunterio/hunter_api.py'>

# Load data

In [10]:
#normalize = False
normalize = True
df = hyamm.get_data_from_hedge_fund_list(normalize)
print("shape=", df.shape)
df.head(2)

INFO  Loading cached version from disk ...
INFO  Loading cached version from disk done (0.009 s)
shape= (666, 18)


,hash,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes
0,,hedge_fund_list,,Alan,Samdperil,asamdperil@aol.com,,,,,,,"Palm Desert, CA, US",,,,hedge_fund,
1,,hedge_fund_list,,Dave,Frevert,frevertdav@aol.com,,,,,,,"Merrit Island, FL, US",,,,hedge_fund,


In [11]:
contact_df = df

In [12]:
contact_df = hyamm.clean_up_contact_df(contact_df)

duplicated_emails: removed 2 / 666 = 0.30%
remove_invalid_emails: removed 15 / 664 = 2.26%
remove_chinese_names: removed 0 / 649 = 0.00%
remove_empty_first_names: removed 7 / 649 = 1.08%
clean_company_names: replaced urls in company_name 0 / 642 = 0.00%
clean_linkedin_nans: replaced nans in linkedin_url 0 / 642 = 0.00%
clean_linkedin_emails: replaced emails in linkedin_url 0 / 642 = 0.00%
clean_linkedin_websites: replaced urls in linkedin_url 0 / 642 = 0.00%
clean_emails: replaced emails 0 / 642 = 0.00%


In [13]:
#mode = "AssumeEmailValid"
mode = "FromScratch"
#enrich_kwargs = {"issue_warnings": False}
enrich_kwargs = {}
contact_df_tmp = cmhuhuap.process_enrich(contact_df, "first_name", "last_name", "company_name", mode=mode)
display(contact_df_tmp.head(2))

  0%|          | 0/642 [00:00<?, ?it/s]

hits=642 misses=0 tot=642 hit_rate=1.00


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,hunterio.company_name,hunterio.company_domain,hunterio.email,hunterio.linkedin_url,hunterio.job_title
hash,,,,,,,,,,,,,,,,,,,,,,
003075018c71251c1de74dbfc2cec0fd,hedge_fund_list,,Kevin,Harper,kharper@downtown-associates.com,,,,,"Downtown Associates, LLC",,"Kennett Square, PA, US",,,,hedge_fund,,_nan_,saultdowntown.com,_nan_,_nan_,_nan_
005fc274f62fee2a3f16ae1b6d9a9277,hedge_fund_list,,Gary,Schwendiman,hilary@schpartners.com,,,,,"Schwendiman International IT Cayman Fund, Ltd",,", , US",,,,hedge_fund,,_nan_,_nan_,_nan_,_nan_,_nan_


In [14]:
hyamm.flush_cache_to_disk()

INFO  Before:
  enrich:
    memory: 7670
    disk: 7670
  find_email:
    memory: -
    disk: 3740
  verify_email:
    memory: -
    disk: 5215
INFO  After:
  enrich:
    memory: 7670
    disk: 7670
  find_email:
    memory: 3740
    disk: 3740
  verify_email:
    memory: 5215
    disk: 5215


In [15]:
contact_df_enriched = cmhuhuap.merge_hunterio_values(contact_df)

In [18]:
contact_df_enriched = hpandas.filter_df(contact_df_enriched, "email", "_nan_", invert=True, check_value=False)

INFO  selected=642 / 642 = 100.00%


In [19]:
hyamm.sanity_check_contact_df(contact_df_enriched)

WARNING All columns must be in Contact schema
{'nan': 1.0}


In [20]:
contact_df_enriched.head(2)

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes
hash,,,,,,,,,,,,,,,,,
003075018c71251c1de74dbfc2cec0fd,hedge_fund_list,,Kevin,Harper,kharper@downtown-associates.com,,,,,"Downtown Associates, LLC",,"Kennett Square, PA, US",,,,hedge_fund,
005fc274f62fee2a3f16ae1b6d9a9277,hedge_fund_list,,Gary,Schwendiman,hilary@schpartners.com,,,,,"Schwendiman International IT Cayman Fund, Ltd",,", , US",,,,hedge_fund,


In [21]:
mode = "AssumeEmailValid"
contact_df_enriched = cmhuhuap.process_email(contact_df_enriched, "first_name", "last_name", "company_name", mode=mode, is_company=True)

contact_df_enriched["email_verification"] = contact_df_enriched["hunterio.email_verification"]

  0%|          | 0/642 [00:00<?, ?it/s]

In [22]:
contact_df_enriched

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,hunterio.email,hunterio.email_verification
hash,,,,,,,,,,,,,,,,,,,
003075018c71251c1de74dbfc2cec0fd,hedge_fund_list,,Kevin,Harper,kharper@downtown-associates.com,invalid,,,,"Downtown Associates, LLC",,"Kennett Square, PA, US",,,,hedge_fund,,kharper@downtown-associates.com,invalid
005fc274f62fee2a3f16ae1b6d9a9277,hedge_fund_list,,Gary,Schwendiman,hilary@schpartners.com,invalid,,,,"Schwendiman International IT Cayman Fund, Ltd",,", , US",,,,hedge_fund,,hilary@schpartners.com,invalid
00631fbcb9589ce2d16f621f48b27398,hedge_fund_list,,William W. A.,Fike,eaglecaptl@aol.com,accept_all,,,,Eagle Capital International,,"Greenwich, CT, US",,,,hedge_fund,,eaglecaptl@aol.com,accept_all
00af2cbbafbce7a73fe9afdcadf360c0,hedge_fund_list,,Eugene,Major,dunbarcm@aol.com,accept_all,,,,"Dunbar Capital Management, LLC",,", , US",,,,hedge_fund,,dunbarcm@aol.com,accept_all
01ac1780e79aaedc7f3948b5d3376d7e,hedge_fund_list,,David,Kreinces,dkreinces@egrowthfund.com,invalid,,,,"E-Growth Fund, L.P.",,", , US",,,,hedge_fund,,dkreinces@egrowthfund.com,invalid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fd3a6c61716413b5096a348e407fdf75,hedge_fund_list,,Ray,Garea,rayg@msfi.com,invalid,,,,Franklin Mutual Advisors,,"Short Hills, NJ, US",,,,hedge_fund,,rayg@msfi.com,invalid
fd733210696b1e70126cd838ef7fb524,hedge_fund_list,,Gary,di Silvestri,gdisilvestri@dsam.com,accept_all,,,,di Silvestri Asset Management (dSAM),,"Stamford, CT, US",,,,hedge_fund,,gdisilvestri@dsam.com,accept_all
fdc38e19fd1d31cbc2d2799750bdf131,hedge_fund_list,,C. P.,Bouckley,rosalyn@bayard-partners.co.uk,invalid,,,,Bayard Fund,,", , US",,,,hedge_fund,,rosalyn@bayard-partners.co.uk,invalid


In [23]:
hyamm.save_to_gsheet(contact_df_enriched, name="hedge_funds")

https://docs.google.com/spreadsheets/d/1HtqneTnbXwqf0xYBjffoqxcEsRXS-Qst4OXCqexLP1I
INFO  Saved to hedge_funds


# Diagnostics

In [22]:
assert 0

AssertionError: 

In [260]:
diagnostics_df = cmhuhuap.get_diagnostic_df(contact_df_tmp)
display(diagnostics_df.head(2))

stats_dict = cmhuhuap.get_stats(diagnostics_df)

if False:
    import pprint
    pprint.pprint(stats_dict, sort_dicts=False)

df_tmp = cmhuhuap.get_stats_df(stats_dict)
display(df_tmp)

,email,hunterio_email,is_email_changed,email_verification,hunterio_email_verification,is_email_verification_changed,first_name,last_name,company_name
292,kharper@downtown-associates.com,kharper@downtown-associates.com,False,,invalid,True,Kevin,Harper,"Downtown Associates, LLC"
594,hilary@schpartners.com,hilary@schpartners.com,False,,invalid,True,Gary,Schwendiman,"Schwendiman International IT Cayman Fund, Ltd"


,tokens [%],non-empty [%],,!reachable,None,_error_,_nan_,accept_all,disposable,invalid,unknown,valid
email,0.00%,100.00%,100.00%,0.00%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hunterio_email,0.00%,100.00%,NaN,35.36%,0.47%,0.16%,4.98%,26.64%,0.16%,50.93%,7.94%,8.72%


In [168]:
hyamm.save_to_gsheet(contact_df_tmp, name="hedge_fund_updated")

https://docs.google.com/spreadsheets/d/1o-vnMd15wzArQBJqKCuMW4Srjd3IstjVBpbKS6K7iT0
INFO  Saved to hedge_fund_updated
